In [1]:
import tensorflow as tf
import numpy as np

## Tensorflow
### Tensores

Arrays multidimensionales inmutables

In [2]:
print("Tensor de rango 0")
print(tf.constant(4))

print("Tensor de rango 1")
print(tf.constant([1,2,3]))

print("Tensor de rango 2")
print(tf.constant([[1,2,3],[1,2,3],[1,2,3]]))

Tensor de rango 0
tf.Tensor(4, shape=(), dtype=int32)
Tensor de rango 1
tf.Tensor([1 2 3], shape=(3,), dtype=int32)
Tensor de rango 2
tf.Tensor(
[[1 2 3]
 [1 2 3]
 [1 2 3]], shape=(3, 3), dtype=int32)



### Variables

Tensores cuyo valor puede ser modificado.

In [3]:
v = tf.Variable([1.,2.])
print (v)
v.assign([3., 4.])
print(v)

<tf.Variable 'Variable:0' shape=(2,) dtype=float32, numpy=array([1., 2.], dtype=float32)>
<tf.Variable 'Variable:0' shape=(2,) dtype=float32, numpy=array([3., 4.], dtype=float32)>


Podemos operar con ellos "normalmente".

"Normalmente" por la aplicación del concepto de broadcasting donde tensores más pequeños son redimensionados/replicados para poder operar con tensores más grandes.

El resultado de operar con tensores o variables es un tensor.

In [4]:
a = tf.constant([[1,2]])
b = tf.constant([[1],[2]])
print(a@b)

print(v*v)

tf.Tensor([[5]], shape=(1, 1), dtype=int32)
tf.Tensor([ 9. 16.], shape=(2,), dtype=float32)


Dada la siguiente función

$$f(w) = x_0 + x_1w + x_2w^2$$

para $w=0$ y $x = [100, -20, 2]$

¿Cuál es el valor $f(w)$?

In [5]:
# Tensor variable
w = tf.Variable(0, name='w', dtype=tf.float32)

# Tensor inmutable
f = w**2. -20.*w + 100.
f

<tf.Tensor: shape=(), dtype=float32, numpy=100.0>

## Perceptrón simple: Compuertas AND, OR y XOR

In [6]:
# Entradas a la compuerta
X = np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=np.float32)

y_and = np.array([[0.],[0.],[0.],[1.]], dtype=np.float32)

y_or = np.array([[0.],[1.],[1.],[1.]], dtype=np.float32)

y_xor = np.array([[0.],[1.],[1.],[0.]], dtype=np.float32)

#### Primero intentemos obtener los pesos manualmente. Define los pesos y el bías como constantes.

In [7]:
# Definimos los pesos y bias
W = tf.constant([[1.],[1.]],dtype=tf.float32)
b = tf.constant([-.5],dtype=tf.float32)

# Perceptrón simple (con función de activación sigmoide)
y_ = tf.nn.sigmoid(tf.matmul(X, W, name='matmul') + b, name="y_")

# Resultado
print("Output", np.round(y_.numpy()).reshape(4,))

Output [0. 1. 1. 1.]


#### Probemos a optimizar los pesos automáticament

### Autodiferenciación

¿Cuál es el valor $f'(w)$?


In [8]:
w = tf.Variable(0, name='w', dtype=tf.float32)

with tf.GradientTape() as t:
  f = w**2. -20.*w + 100.

df_dw = t.gradient(f, w)
df_dw.numpy()

np.float32(-20.0)

#### Variables observadas durante el proceso de diferenciación

In [9]:
t.watched_variables()

(<tf.Variable 'w:0' shape=() dtype=float32, numpy=0.0>,)

¿Cuál es valor de $w$ que minimiza el valor de la siguiente función siendo $x = [100, -20, 2]$?

$$f(w)=x_0 + x_1w + x_2w^2$$

#### Solución
$w=10$

In [10]:
# Coeficientes a optimizar
w = tf.Variable(0, name='w', dtype=tf.float32)

#Definimos el optimizador (SGD)
optimizer = tf.optimizers.SGD(learning_rate=.1)

@tf.function
def train_step():
    with tf.GradientTape() as tape:
        # Se registran las funciones a optimizar
        f = w**2. -20.*w + 100.
    #Obtenemos los gradientes
    gradients = tape.gradient(f, [w])
    #Optimizamos (1 paso)
    optimizer.apply_gradients(zip(gradients, [w]))

In [11]:
for i in range(100):
    train_step()
    print(f"iterations: {i},\tf: {w**2. -20.*w + 100},\tw: {w.numpy()}")


iterations: 0,	f: 64.0,	w: 2.0
iterations: 1,	f: 40.959999084472656,	w: 3.5999999046325684
iterations: 2,	f: 26.214393615722656,	w: 4.880000114440918
iterations: 3,	f: 16.7772216796875,	w: 5.904000282287598
iterations: 4,	f: 10.737419128417969,	w: 6.72320032119751
iterations: 5,	f: 6.8719482421875,	w: 7.3785600662231445
iterations: 6,	f: 4.398048400878906,	w: 7.902848243713379
iterations: 7,	f: 2.8147430419921875,	w: 8.32227897644043
iterations: 8,	f: 1.8014450073242188,	w: 8.65782356262207
iterations: 9,	f: 1.152923583984375,	w: 8.92625904083252
iterations: 10,	f: 0.7378692626953125,	w: 9.141007423400879
iterations: 11,	f: 0.47223663330078125,	w: 9.312806129455566
iterations: 12,	f: 0.3022308349609375,	w: 9.450244903564453
iterations: 13,	f: 0.19342803955078125,	w: 9.560195922851562
iterations: 14,	f: 0.12380218505859375,	w: 9.648157119750977
iterations: 15,	f: 0.07923126220703125,	w: 9.718525886535645
iterations: 16,	f: 0.05071258544921875,	w: 9.774820327758789
iterations: 17,	f: 0.0

#### Continuemos con la neurona artificial

In [18]:
# Definimos los pesos, ahora son variables porque tenemos que optimizarlos
W = tf.Variable(tf.random.normal(shape=[2,1],stddev=np.sqrt(1/2)),name="w",dtype= tf.float32)
b = tf.Variable(tf.zeros([1]),name="b",dtype= tf.float32)

#Definimos las métricas
cost = tf.metrics.Mean(name='cost')
accuracy = tf.metrics.BinaryAccuracy(name='accuracy')

#Definimos el optimizador (SGD)
optimizer = tf.optimizers.SGD(learning_rate=1.)

@tf.function
def train_step(X, y):
    with tf.GradientTape() as tape:
        # Se registran las funciones a optimizar
        y_ = tf.nn.sigmoid(tf.matmul(X, W, name='matmul') + b, name="y_")
        # Usar binary_crossentropy
        loss = tf.losses.binary_crossentropy(y, y_)
    #Obtenemos los gradientes
    gradients = tape.gradient(loss,(W,b))
    #Optimizamos (1 paso)
    optimizer.apply_gradients(zip(gradients, (W,b)))

    #Calculamos las métricas
    cost(loss)
    accuracy(y, y_)

Probemos el funcionamiento de nuetro perceptron simple con las tres compuertas
#### AND

In [19]:
EPOCHS = 20

for epoch in range(EPOCHS):
    # Reset the metrics at the start of the next epoch
    cost.reset_state()
    accuracy.reset_state()

    train_step(X, y_and)

    template = 'Epoch {}, Cost: {}, Accuracy: {}'
    print(template.format(epoch+1,
                        cost.result(),
                        accuracy.result()*100))

y_ = tf.nn.sigmoid(tf.matmul(X, W, name='matmul') + b, name="y_")
print("Output", np.round(y_.numpy()).reshape(4,))

Epoch 1, Cost: 0.6973893642425537, Accuracy: 50.0
Epoch 2, Cost: 0.5740928649902344, Accuracy: 75.0
Epoch 3, Cost: 0.49389344453811646, Accuracy: 75.0
Epoch 4, Cost: 0.4371718168258667, Accuracy: 75.0
Epoch 5, Cost: 0.39373552799224854, Accuracy: 100.0
Epoch 6, Cost: 0.3592189848423004, Accuracy: 100.0
Epoch 7, Cost: 0.33102139830589294, Accuracy: 100.0
Epoch 8, Cost: 0.30746209621429443, Accuracy: 100.0
Epoch 9, Cost: 0.2874132990837097, Accuracy: 100.0
Epoch 10, Cost: 0.2700897753238678, Accuracy: 100.0
Epoch 11, Cost: 0.25492918491363525, Accuracy: 100.0
Epoch 12, Cost: 0.24151857197284698, Accuracy: 100.0
Epoch 13, Cost: 0.2295476645231247, Accuracy: 100.0
Epoch 14, Cost: 0.21877852082252502, Accuracy: 100.0
Epoch 15, Cost: 0.20902544260025024, Accuracy: 100.0
Epoch 16, Cost: 0.20014089345932007, Accuracy: 100.0
Epoch 17, Cost: 0.19200614094734192, Accuracy: 100.0
Epoch 18, Cost: 0.1845242828130722, Accuracy: 100.0
Epoch 19, Cost: 0.1776152104139328, Accuracy: 100.0
Epoch 20, Cost:

#### OR
Ejecutar primero el bloque donde se definen los pesos para reinicializar los valores

In [20]:
EPOCHS = 20

for epoch in range(EPOCHS):
    # Reset the metrics at the start of the next epoch
    cost.reset_state()
    accuracy.reset_state()

    train_step(X, y_or)

    template = 'Epoch {}, Cost: {}, Accuracy: {}'
    print(template.format(epoch+1,
                        cost.result(),
                        accuracy.result()*100))

y_ = tf.nn.sigmoid(tf.matmul(X, W, name='matmul') + b, name="y_")
print("Output", np.round(y_.numpy()).reshape(4,))

Epoch 1, Cost: 0.9622243642807007, Accuracy: 50.0
Epoch 2, Cost: 0.1354379951953888, Accuracy: 100.0
Epoch 3, Cost: 0.09851999580860138, Accuracy: 100.0
Epoch 4, Cost: 0.08651698380708694, Accuracy: 100.0
Epoch 5, Cost: 0.08057549595832825, Accuracy: 100.0
Epoch 6, Cost: 0.07678045332431793, Accuracy: 100.0
Epoch 7, Cost: 0.07390803098678589, Accuracy: 100.0
Epoch 8, Cost: 0.0714973658323288, Accuracy: 100.0
Epoch 9, Cost: 0.0693538561463356, Accuracy: 100.0
Epoch 10, Cost: 0.06738778948783875, Accuracy: 100.0
Epoch 11, Cost: 0.06555410474538803, Accuracy: 100.0
Epoch 12, Cost: 0.06382807344198227, Accuracy: 100.0
Epoch 13, Cost: 0.0621945746243, Accuracy: 100.0
Epoch 14, Cost: 0.06064344197511673, Accuracy: 100.0
Epoch 15, Cost: 0.05916714295744896, Accuracy: 100.0
Epoch 16, Cost: 0.057759687304496765, Accuracy: 100.0
Epoch 17, Cost: 0.056415997445583344, Accuracy: 100.0
Epoch 18, Cost: 0.055131714791059494, Accuracy: 100.0
Epoch 19, Cost: 0.053902920335531235, Accuracy: 100.0
Epoch 2

#### XOR
Ejecutar primero el bloque donde se definen los pesos para reinicializar los valores

In [21]:
EPOCHS = 20

for epoch in range(EPOCHS):
    # Reset the metrics at the start of the next epoch
    cost.reset_state()
    accuracy.reset_state()

    train_step(X, y_xor)

    template = 'Epoch {}, Cost: {}, Accuracy: {}'
    print(template.format(epoch+1,
                        cost.result(),
                        accuracy.result()*100))

y_ = tf.nn.sigmoid(tf.matmul(X, W, name='matmul') + b, name="y_")
print("Output", np.round(y_.numpy()).reshape(4,))

Epoch 1, Cost: 2.116342306137085, Accuracy: 75.0
Epoch 2, Cost: 1.4861171245574951, Accuracy: 75.0
Epoch 3, Cost: 1.232041835784912, Accuracy: 25.0
Epoch 4, Cost: 1.1509207487106323, Accuracy: 25.0
Epoch 5, Cost: 1.0846657752990723, Accuracy: 25.0
Epoch 6, Cost: 1.0231693983078003, Accuracy: 25.0
Epoch 7, Cost: 0.9668188095092773, Accuracy: 25.0
Epoch 8, Cost: 0.9161820411682129, Accuracy: 25.0
Epoch 9, Cost: 0.8716790676116943, Accuracy: 25.0
Epoch 10, Cost: 0.8335093259811401, Accuracy: 25.0
Epoch 11, Cost: 0.8016035556793213, Accuracy: 25.0
Epoch 12, Cost: 0.7756166458129883, Accuracy: 25.0
Epoch 13, Cost: 0.7549703121185303, Accuracy: 25.0
Epoch 14, Cost: 0.7389345169067383, Accuracy: 25.0
Epoch 15, Cost: 0.7267225384712219, Accuracy: 25.0
Epoch 16, Cost: 0.717573881149292, Accuracy: 25.0
Epoch 17, Cost: 0.7108097076416016, Accuracy: 25.0
Epoch 18, Cost: 0.7058596014976501, Accuracy: 25.0
Epoch 19, Cost: 0.7022653818130493, Accuracy: 25.0
Epoch 20, Cost: 0.6996707916259766, Accurac

#### ¿Qué ha sucedido con la compuerta XOR?

## Perceptrón Multicapa: Compuerta XOR
Crear un perceptron multicapa con topología 2-2-1 (2 entradas, 2 ocultas y una salida)

In [ ]:
# Pesos del perceptrón multicapa
Wh = #TODO
bh = #TODO

Wo = #TODO
bo = #TODO

#Definimos las métricas
cost = #TODO
accuracy = #TODO

#Definimos el optimizador (SGD)
optimizer = #TODO

@tf.function
def train_step(X, y):
    with tf.GradientTape() as tape:
        # Se registran las funciones a optimizar
        # Perceptrón multicapa
        #Capa oculta
        a_h = #TODO
        #Capa de salida
        y_ = #TODO
        # Usar binary_crossentropy
        loss = #TODO
    #Obtenemos los gradientes
    gradients = #TODO
    #Optimizamos (1 paso)
    optimizer.#TODO

    #Calculamos las métricas
    cost(#TODO
    accuracy(#TODO

Ejecuta varias veces junto al anterior, no encuentra siempre la solución

In [ ]:
EPOCHS = 100

for epoch in range(EPOCHS):
    # Reset the metrics at the start of the next epoch
    cost.reset_state()
    accuracy.reset_state()

    train_step(X, y_xor)
    if epoch%10==0:
        template = 'Epoch {}, Cost: {}, Accuracy: {}'
        print(template.format(epoch+1,
                            cost.result(),
                            accuracy.result()*100))

y_ = tf.nn.sigmoid(tf.matmul(tf.nn.tanh(tf.matmul(X, Wh) + bh, name='a_h'), Wo) + bo, name='y_')
print("Output", np.round(y_.numpy()).reshape(4,))

Epoch 1, Cost: 0.3509092330932617, Accuracy: 50.0
Epoch 11, Cost: 0.3504387140274048, Accuracy: 50.0
Epoch 21, Cost: 0.3500578999519348, Accuracy: 50.0
Epoch 31, Cost: 0.3497440218925476, Accuracy: 50.0
Epoch 41, Cost: 0.3494805693626404, Accuracy: 50.0
Epoch 51, Cost: 0.3492566645145416, Accuracy: 50.0
Epoch 61, Cost: 0.3490641117095947, Accuracy: 50.0
Epoch 71, Cost: 0.3488968014717102, Accuracy: 50.0
Epoch 81, Cost: 0.3487498164176941, Accuracy: 50.0
Epoch 91, Cost: 0.3486202359199524, Accuracy: 50.0
Output [0. 0. 1. 1.]
